In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import random

spark = SparkSession.builder.appName("DataGenerator").getOrCreate()

# Sample values
products = ["Laptop", "Mobile", "Tablet", "Watch", "Headphones"]
categories = ["Electronics", "Accessories"]
cities = ["Hyderabad", "Chennai", "Bangalore", "Mumbai", "Delhi", None]

data = []

for i in range(20000):
    order_id = random.randint(1000, 3000)  # duplicates + updates
    customer_id = f"C{random.randint(1, 500)}"
    product = random.choice(products)
    category = random.choice(categories)
    city = random.choice(cities)
    date = f"2024-01-{random.randint(1,28):02d}"
    
    # introduce NULL + negative values
    amount = random.choice([
        random.randint(1000, 60000),
        None,
        -random.randint(1000, 5000)
    ])
    
    quantity = random.randint(1, 5)

    data.append((order_id, customer_id, product, category, city, date, amount, quantity))

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df = spark.createDataFrame(data, columns)


In [0]:
df.write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/sample_dataset/bronze")

In [0]:
df = spark.read.option("header", True) \
    .csv("/Volumes/workspace/default/sample_dataset/bronze")

In [0]:
from pyspark.sql.functions import to_date

df_clean = df.withColumn("amount", col("amount").cast("int")) \
            .withColumn("order_id", col("order_id").cast("int"))\
       .withColumn("date", to_date(col("date")))

In [0]:
df_clean = df_clean.filter(col("amount") > 0)

In [0]:
 from pyspark.sql.window import Window
window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_clean = df_clean.withColumn("rank", row_number().over(window_spec)) \
             .filter(col("rank") == 1) \
             .drop("rank")
df_clean.display()

order_id,customer_id,product,category,city,date,amount,quantity
1000,C260,Tablet,Electronics,null,2024-01-23,43550,1
1001,C9,Headphones,Accessories,Delhi,2024-01-16,51177,5
1002,C47,Tablet,Electronics,Delhi,2024-01-28,30253,4
1003,C131,Watch,Accessories,Mumbai,2024-01-15,39905,4
1004,C306,Mobile,Accessories,Hyderabad,2024-01-17,25985,3
1005,C468,Mobile,Electronics,Chennai,2024-01-28,40441,4
1006,C114,Laptop,Electronics,Chennai,2024-01-14,19932,1
1007,C70,Watch,Accessories,Delhi,2024-01-03,17215,5
1008,C278,Laptop,Electronics,Hyderabad,2024-01-25,23543,2
1009,C36,Laptop,Electronics,null,2024-01-20,35991,1


In [0]:
df_clean.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/sample_dataset/silver")

In [0]:
from pyspark.sql.functions import sum, col

df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/sample_dataset/silver")

Total Sales by Product

In [0]:
sales_product = df.groupBy("product") \
    .agg(sum("amount").alias("total_sales"))

display(sales_product)

product,total_sales
Tablet,11333584
Mobile,11281493
Watch,13509658
Headphones,11328821
Laptop,11274747


Total Sales by City

In [0]:
sales_city = df.groupBy("city") \
    .agg(sum("amount").alias("total_sales"))
display(sales_city)

city,total_sales
Delhi,10430188
Chennai,9311419
Hyderabad,9339615
Bangalore,10057459
null,10395313
Mumbai,9194309


Total Revenue

In [0]:
total_revenue = df.agg(sum("amount").alias("total_revenue"))

display(total_revenue)

total_revenue
58728303


Top Customer (Highest Spending)

In [0]:
customer_spending = df.groupBy("customer_id") \
    .agg(sum("amount").alias("total_spent"))

top_customer = customer_spending.orderBy(col("total_spent").desc()).limit(1)

display(top_customer)

customer_id,total_spent
C91,378555


Top Product

In [0]:
top_product = sales_product.orderBy(col("total_sales").desc()).limit(1)

display(top_product)

product,total_sales
Watch,13509658


In [0]:
sales_product.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/sample_dataset/gold/sales_product")

sales_city.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/sample_dataset/gold/sales_city")
total_revenue.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/sample_dataset/gold/total_revenue")
customer_spending.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/sample_dataset/gold/customer_spending")
top_product.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/sample_dataset/gold/top_product")